# Etapa 3 - Preparación y limpieza

Ejecutar todas las celdas en orden desde la raíz del repositorio o `notebooks/`. Requiere pandas, openpyxl e IPython. Genera únicamente el Excel procesado y el informe de esta etapa; una nueva ejecución regenera esos entregables desde el original. No modifica los archivos de etapas anteriores. Conserva filas, orden, columnas, valores extremos y NA; aplica exclusivamente las dos sustituciones exactas autorizadas.

In [1]:
from pathlib import Path
import hashlib
import platform
import pandas as pd
import openpyxl
from IPython.display import display, Markdown

root = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "dataset_salud_publica_2500.xlsx").is_file()
            and (p / "reports/02_auditoria_dataset.md").is_file())
source = root / "dataset_salud_publica_2500.xlsx"
target = root / "data/processed/dataset_salud_publica_limpio.xlsx"
protected = [source, root/"reports/01_objetivos.md",
             root/"reports/02_auditoria_dataset.md", root/"notebooks/02_auditoria.ipynb"]
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()
hashes_before = {p: sha256(p) for p in protected}
previous_reports = {p.name:p.read_text(encoding="utf-8") for p in protected[1:3]}
original = pd.read_excel(source, sheet_name="Salud Publica", engine="openpyxl")
assert original.shape == (2500, 10)
assert source.resolve() != target.resolve()
print("Informes previos leídos. Original cargado: 2500 filas y 10 columnas.")


Informes previos leídos. Original cargado: 2500 filas y 10 columnas.


## Copia y normalizaciones autorizadas

No se utiliza normalización general de texto. Las sustituciones son exactas y se limitan a las dos columnas autorizadas.

In [2]:
rules = {"Condicion_Salud": ("Cronica", "Crónica"),
         "Cobertura_Salud": ("Publica", "Pública")}
clean = original.copy(deep=True)
counts = {}
for column, (before, after) in rules.items():
    counts[column] = int(original[column].eq(before).sum())
    clean[column] = clean[column].replace({before: after})

# Verificación previa a exportar: todos los NA permanecen en su celda.
pd.testing.assert_frame_equal(original.isna(), clean.isna(), check_exact=True)
unchanged_columns = [c for c in original.columns if c not in rules]
pd.testing.assert_frame_equal(original[unchanged_columns], clean[unchanged_columns], check_exact=True)
print("Sustituciones exactas:", counts)


Sustituciones exactas: {'Condicion_Salud': 843, 'Cobertura_Salud': 806}


## Exportación y lectura del archivo resultante

Los NA se escriben como celdas vacías de Excel y se recuperan como NA al cargar con pandas. No se exporta el índice como columna.

In [3]:
target.parent.mkdir(parents=True, exist_ok=True)
clean.to_excel(target, sheet_name="Salud Publica", index=False, engine="openpyxl")
processed = pd.read_excel(target, sheet_name="Salud Publica", engine="openpyxl")
pd.testing.assert_frame_equal(clean, processed, check_exact=True, check_dtype=True)
print("Archivo procesado guardado y vuelto a leer correctamente.")


Archivo procesado guardado y vuelto a leer correctamente.


## Validaciones del archivo guardado

La comparación se realiza celda por celda, preservando el orden original y considerando iguales los NA coincidentes. La máscara de cambios debe coincidir exactamente con la máscara de las dos sustituciones autorizadas, incluyendo sus valores de destino.

In [4]:
validations = []
def check(label, actual, expected):
    assert actual == expected, f"{label}: {actual!r} != {expected!r}"
    validations.append([label, str(expected), str(actual), "OK"])

check("Filas", len(processed), 2500)
check("Columnas", len(processed.columns), 10)
check("ID_Paciente únicos", processed.ID_Paciente.nunique(), 2500)
check("IDs nulos", int(processed.ID_Paciente.isna().sum()), 0)
check("Filas completamente duplicadas", int(processed.duplicated().sum()), 0)
check("NA en Tiempo_Espera_min", int(processed.Tiempo_Espera_min.isna().sum()), 74)
check("NA en Satisfaccion", int(processed.Satisfaccion.isna().sum()), 72)
for c, (old, new) in rules.items():
    check(f"Apariciones de {old}", int(processed[c].eq(old).sum()), 0)
    check(f"Presencia de {new}", bool(processed[c].eq(new).any()), True)
    check(f"Apariciones de {new}", int(processed[c].eq(new).sum()),
          counts[c] + int(original[c].eq(new).sum()))

pd.testing.assert_index_equal(original.columns, processed.columns, exact=True)
pd.testing.assert_index_equal(original.index, processed.index, exact=True)
pd.testing.assert_series_equal(original.ID_Paciente, processed.ID_Paciente, check_exact=True)
pd.testing.assert_frame_equal(original.isna(), processed.isna(), check_exact=True)
pd.testing.assert_frame_equal(original[unchanged_columns], processed[unchanged_columns], check_exact=True)
same = original.eq(processed).fillna(False) | (original.isna() & processed.isna())
changed = ~same
allowed = pd.DataFrame(False, index=original.index, columns=original.columns)
for c, (old, new) in rules.items():
    allowed[c] = original[c].eq(old).fillna(False)
    assert processed.loc[allowed[c], c].eq(new).all()
assert (changed == allowed).to_numpy().all(), "Hay cambios faltantes o no autorizados."
check("Celdas modificadas autorizadas", int(changed.to_numpy().sum()), sum(counts.values()))
check("Celdas modificadas no autorizadas", int((changed & ~allowed).to_numpy().sum()), 0)
check("Cambios en ubicación de NA", int((original.isna() != processed.isna()).to_numpy().sum()), 0)
for p in protected:
    check(f"SHA-256 intacto: {p.relative_to(root).as_posix()}", sha256(p), hashes_before[p])
print("Todas las validaciones de estructura, valores, NA e integridad fueron aprobadas.")


Todas las validaciones de estructura, valores, NA e integridad fueron aprobadas.


## Informe reproducible

Las tablas se generan con columnas alineadas para su lectura en Markdown y GitHub. Los conteos se obtienen del archivo exportado y de su comparación con el original.

In [5]:
def table(headers, rows, numeric=()):
    data = [[str(v) for v in row] for row in rows]
    widths = [max(3, len(h), *(len(r[j]) for r in data)) for j,h in enumerate(headers)]
    lines = ["| " + " | ".join(h.ljust(widths[j]) for j,h in enumerate(headers)) + " |"]
    lines.append("| " + " | ".join("-"*(w-1)+":" if j in numeric else "-"*w for j,w in enumerate(widths)) + " |")
    lines += ["| " + " | ".join(v.rjust(widths[j]) if j in numeric else v.ljust(widths[j]) for j,v in enumerate(r)) + " |" for r in data]
    return "\n".join(lines)

transformations = table(["Variable","Antes","Después","Motivo"],
    [[c,old,new,"Consistencia con documentación, gráficos y presentación."] for c,(old,new) in rules.items()])
change_counts = table(["Variable","Celdas modificadas"], [[c,counts[c]] for c in rules], numeric=(1,))
missing_rows = [[c,int(original[c].isna().sum()),int(processed[c].isna().sum())]
                for c in ["Tiempo_Espera_min","Satisfaccion"]]
missing_table = table(["Variable","NA originales","NA conservados"], missing_rows, numeric=(1,2))
checks_table = table(["Validación","Esperado","Obtenido","Resultado"],
                     [row for row in validations if not row[0].startswith("SHA-256")])
hash_table = table(["Archivo protegido","SHA-256 antes y después"],
                   [[p.relative_to(root).as_posix(), hashes_before[p]] for p in protected])
report = f"""# Etapa 3 - Preparación y limpieza

## 1. Decisiones adoptadas

Se aplicaron las decisiones del equipo sobre la base de reports/01_objetivos.md y reports/02_auditoria_dataset.md: conservar los 2500 registros, sin eliminar filas ni imputar faltantes, y normalizar exclusivamente las dos categorías indicadas. Se preservaron el orden de las filas, las diez columnas, los identificadores y los valores extremos.

No se crearon grupos etarios, variables derivadas, indicadores, variables de vulnerabilidad o mala atención ni targets. Esta etapa no incluye EDA, gráficos, análisis de relaciones ni modelos.

## 2. Transformaciones realizadas

Las sustituciones se aplicaron por coincidencia exacta sobre una copia en memoria del original.

{transformations}

{change_counts}

Se modificaron **{sum(counts.values())} celdas** en total. Este conteo corresponde a celdas, no necesariamente a pacientes distintos. El resto de las categorías permaneció intacto.

## 3. Tratamiento de valores faltantes

**Los valores faltantes se conservaron como NA, sin imputación ni eliminación de filas.** En el Excel se almacenan como celdas vacías y se recuperan como NA mediante pandas.

{missing_table}

Se preservaron las {int(processed.isna().sum().sum())} celdas faltantes en sus posiciones originales: {int(processed.isna().any(axis=1).sum())} filas tienen al menos un NA y {int((processed.isna().sum(axis=1)>1).sum())} tienen ambos. No se introdujeron faltantes nuevos.

**En los análisis posteriores se utilizarán casos disponibles según las variables involucradas y se informará el N efectivo utilizado en cada análisis.** Esta decisión no implica eliminar registros del dataset procesado.

## 4. Validaciones

Los controles se ejecutaron sobre el archivo procesado después de guardarlo y volverlo a cargar. Todas las comprobaciones se implementaron con aserciones.

{checks_table}

También se verificó igualdad exacta de valores y tipos detectados por pandas entre la copia preparada en memoria y el archivo vuelto a cargar.

## 5. Integridad respecto del dataset original

Se compararon programáticamente las 25000 celdas del original y del procesado, en el mismo orden, considerando iguales los NA coincidentes. La máscara de diferencias coincidió exactamente con las posiciones originales de Cronica en Condicion_Salud y Publica en Cobertura_Salud; se comprobaron además sus destinos Crónica y Pública.

Se verificó igualdad exacta de las otras ocho columnas: ID_Paciente, Region, Edad, Genero, Frecuencia_Atencion, Tiempo_Espera_min, Acceso_Medicacion y Satisfaccion. No cambió ningún otro valor, la ubicación de los NA, el orden de los registros ni los encabezados. Los valores extremos quedaron intactos.

Los cuatro archivos protegidos mantuvieron sus hashes SHA-256 antes y después de la ejecución:

{hash_table}

## 6. Dataset resultante

- **Archivo:** data/processed/dataset_salud_publica_limpio.xlsx.
- **Hoja:** Salud Publica.
- **Dimensiones:** {len(processed)} filas y {len(processed.columns)} columnas, sin columna adicional de índice.
- **Origen preservado:** dataset_salud_publica_2500.xlsx.
- **Notebook reproducible:** notebooks/03_limpieza.ipynb. Ejecutar todas las celdas desde la raíz del repositorio o notebooks/ regenera el procesado y este informe.
- **Entorno de ejecución:** Python {platform.python_version()}, pandas {pd.__version__}, openpyxl {openpyxl.__version__}.
- **SHA-256 del procesado de esta ejecución:** {sha256(target)}.

La preparación y limpieza finaliza con las dos normalizaciones autorizadas y la conservación de todos los registros y NA.
"""
(root/"reports/03_limpieza.md").write_text(report, encoding="utf-8")
display(Markdown(report))


# Etapa 3 - Preparación y limpieza

## 1. Decisiones adoptadas

Se aplicaron las decisiones del equipo sobre la base de reports/01_objetivos.md y reports/02_auditoria_dataset.md: conservar los 2500 registros, sin eliminar filas ni imputar faltantes, y normalizar exclusivamente las dos categorías indicadas. Se preservaron el orden de las filas, las diez columnas, los identificadores y los valores extremos.

No se crearon grupos etarios, variables derivadas, indicadores, variables de vulnerabilidad o mala atención ni targets. Esta etapa no incluye EDA, gráficos, análisis de relaciones ni modelos.

## 2. Transformaciones realizadas

Las sustituciones se aplicaron por coincidencia exacta sobre una copia en memoria del original.

| Variable        | Antes   | Después | Motivo                                                   |
| --------------- | ------- | ------- | -------------------------------------------------------- |
| Condicion_Salud | Cronica | Crónica | Consistencia con documentación, gráficos y presentación. |
| Cobertura_Salud | Publica | Pública | Consistencia con documentación, gráficos y presentación. |

| Variable        | Celdas modificadas |
| --------------- | -----------------: |
| Condicion_Salud |                843 |
| Cobertura_Salud |                806 |

Se modificaron **1649 celdas** en total. Este conteo corresponde a celdas, no necesariamente a pacientes distintos. El resto de las categorías permaneció intacto.

## 3. Tratamiento de valores faltantes

**Los valores faltantes se conservaron como NA, sin imputación ni eliminación de filas.** En el Excel se almacenan como celdas vacías y se recuperan como NA mediante pandas.

| Variable          | NA originales | NA conservados |
| ----------------- | ------------: | -------------: |
| Tiempo_Espera_min |            74 |             74 |
| Satisfaccion      |            72 |             72 |

Se preservaron las 146 celdas faltantes en sus posiciones originales: 143 filas tienen al menos un NA y 3 tienen ambos. No se introdujeron faltantes nuevos.

**En los análisis posteriores se utilizarán casos disponibles según las variables involucradas y se informará el N efectivo utilizado en cada análisis.** Esta decisión no implica eliminar registros del dataset procesado.

## 4. Validaciones

Los controles se ejecutaron sobre el archivo procesado después de guardarlo y volverlo a cargar. Todas las comprobaciones se implementaron con aserciones.

| Validación                        | Esperado | Obtenido | Resultado |
| --------------------------------- | -------- | -------- | --------- |
| Filas                             | 2500     | 2500     | OK        |
| Columnas                          | 10       | 10       | OK        |
| ID_Paciente únicos                | 2500     | 2500     | OK        |
| IDs nulos                         | 0        | 0        | OK        |
| Filas completamente duplicadas    | 0        | 0        | OK        |
| NA en Tiempo_Espera_min           | 74       | 74       | OK        |
| NA en Satisfaccion                | 72       | 72       | OK        |
| Apariciones de Cronica            | 0        | 0        | OK        |
| Presencia de Crónica              | True     | True     | OK        |
| Apariciones de Crónica            | 843      | 843      | OK        |
| Apariciones de Publica            | 0        | 0        | OK        |
| Presencia de Pública              | True     | True     | OK        |
| Apariciones de Pública            | 806      | 806      | OK        |
| Celdas modificadas autorizadas    | 1649     | 1649     | OK        |
| Celdas modificadas no autorizadas | 0        | 0        | OK        |
| Cambios en ubicación de NA        | 0        | 0        | OK        |

También se verificó igualdad exacta de valores y tipos detectados por pandas entre la copia preparada en memoria y el archivo vuelto a cargar.

## 5. Integridad respecto del dataset original

Se compararon programáticamente las 25000 celdas del original y del procesado, en el mismo orden, considerando iguales los NA coincidentes. La máscara de diferencias coincidió exactamente con las posiciones originales de Cronica en Condicion_Salud y Publica en Cobertura_Salud; se comprobaron además sus destinos Crónica y Pública.

Se verificó igualdad exacta de las otras ocho columnas: ID_Paciente, Region, Edad, Genero, Frecuencia_Atencion, Tiempo_Espera_min, Acceso_Medicacion y Satisfaccion. No cambió ningún otro valor, la ubicación de los NA, el orden de los registros ni los encabezados. Los valores extremos quedaron intactos.

Los cuatro archivos protegidos mantuvieron sus hashes SHA-256 antes y después de la ejecución:

| Archivo protegido               | SHA-256 antes y después                                          |
| ------------------------------- | ---------------------------------------------------------------- |
| dataset_salud_publica_2500.xlsx | 06449a4ada23692fe43b173198d4d46e05c28a29b5fe796e72eb6926bd1f5100 |
| reports/01_objetivos.md         | 845f2358abe8bfd478dd8ccdf7ead2a45df38c46fb8332bc664ed0b2853663d5 |
| reports/02_auditoria_dataset.md | e6005884fc9cc6f4059dd7fe20a5ae35aca533f5965e9a76a96b2a91c56ee72e |
| notebooks/02_auditoria.ipynb    | a3c654515d3e3d498d1a5fa8a724a1e630b10776d9b01efcdf9e060cf9b68319 |

## 6. Dataset resultante

- **Archivo:** data/processed/dataset_salud_publica_limpio.xlsx.
- **Hoja:** Salud Publica.
- **Dimensiones:** 2500 filas y 10 columnas, sin columna adicional de índice.
- **Origen preservado:** dataset_salud_publica_2500.xlsx.
- **Notebook reproducible:** notebooks/03_limpieza.ipynb. Ejecutar todas las celdas desde la raíz del repositorio o notebooks/ regenera el procesado y este informe.
- **Entorno de ejecución:** Python 3.12.3, pandas 3.0.5, openpyxl 3.1.5.
- **SHA-256 del procesado de esta ejecución:** 6707074a0d805b2fce33f461c10c9335adcb5ab302d960bc0e963db868b983bd.

La preparación y limpieza finaliza con las dos normalizaciones autorizadas y la conservación de todos los registros y NA.
